In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

In [2]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data" / "processed" / "week-3"

NODE_FILE = DATA_DIR / "graph_node_features.csv"
EDGE_FILE = DATA_DIR / "knn_graph_edges.csv"
TARGET_FILE = DATA_DIR / "graph_targets.csv"
EMBEDDING_FILE = DATA_DIR / "spatial_embeddings.csv"

FINAL_NODE_FILE = DATA_DIR / "final_graph_nodes.csv"
FINAL_EDGE_FILE = DATA_DIR / "final_graph_edges.csv"
FINAL_TARGET_FILE = DATA_DIR / "final_graph_targets.csv"

In [3]:
nodes_df = pd.read_csv(NODE_FILE)
edges_df = pd.read_csv(EDGE_FILE)
targets_df = pd.read_csv(TARGET_FILE)
embeddings_df = pd.read_csv(EMBEDDING_FILE)

print("Node features:", nodes_df.shape)
print("Edges:", edges_df.shape)
print("Targets:", targets_df.shape)
print("Spatial embeddings:", embeddings_df.shape)

Node features: (21613, 18)
Edges: (108065, 3)
Targets: (21613, 2)
Spatial embeddings: (21613, 9)


In [4]:
print("Node feature ID range:")
print(nodes_df["node_id"].min(), nodes_df["node_id"].max())

print("\nEmbedding ID range:")
print(embeddings_df["node_id"].min(), embeddings_df["node_id"].max())

print("\nTarget ID range:")
print(targets_df["node_id"].min(), targets_df["node_id"].max())

Node feature ID range:
0 21612

Embedding ID range:
0 21612

Target ID range:
0 21612


In [5]:
node_ids = set(nodes_df["node_id"])
embedding_ids = set(embeddings_df["node_id"])
target_ids = set(targets_df["node_id"])

print("Nodes missing embeddings:", len(node_ids - embedding_ids))
print("Embeddings without nodes:", len(embedding_ids - node_ids))

print("Nodes missing targets:", len(node_ids - target_ids))
print("Targets without nodes:", len(target_ids - node_ids))

Nodes missing embeddings: 0
Embeddings without nodes: 0
Nodes missing targets: 0
Targets without nodes: 0


In [6]:
final_nodes = nodes_df.merge(embeddings_df,on="node_id",how="inner",suffixes=("", "_embedding"))

print("Final node dataset shape:", final_nodes.shape)

Final node dataset shape: (21613, 26)


In [7]:
assert len(final_nodes) == len(nodes_df)

assert final_nodes["node_id"].nunique() == len(final_nodes)

print("✅ Node feature and embedding merge validated.")

✅ Node feature and embedding merge validated.


In [8]:
print("Final node columns:")
print(final_nodes.columns.tolist())

Final node columns:
['node_id', 'id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'House Age', 'yr_built', 'yr_renovated', 'lat', 'long', 'lat_scaled', 'long_scaled', 'mean_neighbor_distance_scaled', 'median_neighbor_distance_scaled', 'min_neighbor_distance_scaled', 'max_neighbor_distance_scaled', 'std_neighbor_distance_scaled', 'neighbor_count']


In [9]:
print("Edge columns:")
print(edges_df.columns.tolist())

print("\nNumber of edges:", len(edges_df))

Edge columns:
['source', 'target', 'distance_km']

Number of edges: 108065


In [10]:
valid_node_ids = set(final_nodes["node_id"])

invalid_sources = (set(edges_df["source"]) - valid_node_ids)

invalid_targets = (set(edges_df["target"]) - valid_node_ids)

print("Invalid source nodes:", len(invalid_sources))
print("Invalid target nodes:", len(invalid_targets))

Invalid source nodes: 0
Invalid target nodes: 0


In [11]:
self_loops = (edges_df["source"] == edges_df["target"]).sum()

print("Self-loops:", self_loops)

assert self_loops == 0

print("✅ Self-loop validation passed.")

Self-loops: 0
✅ Self-loop validation passed.


In [12]:
print(edges_df["distance_km"].describe())

assert edges_df["distance_km"].notna().all()
assert (edges_df["distance_km"] >= 0).all()

print("✅ Edge distance validation passed.")

count    108065.000000
mean          0.196325
std           0.300451
min           0.000000
25%           0.088956
50%           0.153740
75%           0.229664
max          23.517976
Name: distance_km, dtype: float64
✅ Edge distance validation passed.


In [13]:
print("Target columns:")
print(targets_df.columns.tolist())

targets_df.head()

Target columns:
['node_id', 'price']


,node_id,price
0,0,221900.0
1,1,538000.0
2,2,180000.0
3,3,604000.0
4,4,510000.0


In [14]:
assert set(final_nodes["node_id"]) == set(targets_df["node_id"])

print("✅ Target-node alignment validated.")

✅ Target-node alignment validated.


In [15]:
final_nodes_with_target = final_nodes.merge(targets_df,on="node_id",how="inner")

print("Final node + target shape:", final_nodes_with_target.shape)

Final node + target shape: (21613, 27)


In [16]:
print(targets_df.columns.tolist())

TARGET_COLUMN = "price"

assert TARGET_COLUMN in final_nodes_with_target.columns

print("Target column:", TARGET_COLUMN)

feature_columns = [col for col in final_nodes.columns if col not in ["node_id", TARGET_COLUMN]]

print("Number of node features:", len(feature_columns))

['node_id', 'price']
Target column: price
Number of node features: 25


In [17]:
final_node_features = final_nodes[["node_id"] + feature_columns].copy()

print("Final node feature shape:", final_node_features.shape)

Final node feature shape: (21613, 26)


In [18]:
final_targets = targets_df[["node_id", TARGET_COLUMN]].copy()

print("Final target shape:", final_targets.shape)

Final target shape: (21613, 2)


In [19]:
print("Missing node feature values:", final_node_features.isnull().sum().sum())

print("Missing target values:", final_targets.isnull().sum().sum())

print("Missing edge values:", edges_df.isnull().sum().sum())

assert final_node_features.isnull().sum().sum() == 0
assert final_targets.isnull().sum().sum() == 0
assert edges_df.isnull().sum().sum() == 0

print("✅ Missing-value validation passed.")

Missing node feature values: 0
Missing target values: 0
Missing edge values: 0
✅ Missing-value validation passed.


In [20]:
final_node_features.to_csv(FINAL_NODE_FILE,index=False)

edges_df.to_csv(FINAL_EDGE_FILE,index=False)

final_targets.to_csv(FINAL_TARGET_FILE,index=False)

print("✅ Final graph datasets saved.")

✅ Final graph datasets saved.


In [21]:
saved_nodes = pd.read_csv(FINAL_NODE_FILE)
saved_edges = pd.read_csv(FINAL_EDGE_FILE)
saved_targets = pd.read_csv(FINAL_TARGET_FILE)

print("Saved nodes:", saved_nodes.shape)
print("Saved edges:", saved_edges.shape)
print("Saved targets:", saved_targets.shape)

Saved nodes: (21613, 26)
Saved edges: (108065, 3)
Saved targets: (21613, 2)


In [22]:
K = 5

assert saved_nodes["node_id"].nunique() == len(saved_nodes)

assert set(saved_nodes["node_id"]) == set(saved_targets["node_id"])

assert saved_edges["source"].isin(saved_nodes["node_id"]).all()

assert saved_edges["target"].isin(saved_nodes["node_id"]).all()

assert not (saved_edges["source"] == saved_edges["target"]).any()

# Validate expected number of KNN edges
expected_edges = len(saved_nodes) * K

assert len(saved_edges) == expected_edges, (
    f"Expected {expected_edges} edges, "
    f"but found {len(saved_edges)}."
)

# Validate that every node has exactly K outgoing edges
outgoing_degree = (saved_edges.groupby("source").size())

assert len(outgoing_degree) == len(saved_nodes)

assert (outgoing_degree == K).all()

print("✅ Node ID validation passed.")
print("✅ Target alignment validation passed.")
print("✅ Edge reference validation passed.")
print("✅ Self-loop validation passed.")
print(f"✅ Edge count validation passed: {len(saved_edges):,} edges.")
print("✅ Every node has exactly 5 outgoing KNN edges.")
print("✅ Final graph dataset validation passed.")

✅ Node ID validation passed.
✅ Target alignment validation passed.
✅ Edge reference validation passed.
✅ Self-loop validation passed.
✅ Edge count validation passed: 108,065 edges.
✅ Every node has exactly 5 outgoing KNN edges.
✅ Final graph dataset validation passed.


In [23]:
print("=" * 60)
print("WEEK 3 DAY 4 – GRAPH DATASET SUMMARY")
print("=" * 60)

print(f"Graph nodes: {len(saved_nodes):,}")
print(f"Graph edges: {len(saved_edges):,}")
print(f"Node features: {saved_nodes.shape[1] - 1}")
print(f"Targets: {len(saved_targets):,}")

print("\nValidation:")
print("✓ Node IDs aligned")
print("✓ Target IDs aligned")
print("✓ Edge references valid")
print("✓ No self-loops")
print("✓ No missing values")
print("✓ Graph dataset ready")

print("\n✅ Week 3 Day 4 completed successfully.")

WEEK 3 DAY 4 – GRAPH DATASET SUMMARY
Graph nodes: 21,613
Graph edges: 108,065
Node features: 25
Targets: 21,613

Validation:
✓ Node IDs aligned
✓ Target IDs aligned
✓ Edge references valid
✓ No self-loops
✓ No missing values
✓ Graph dataset ready

✅ Week 3 Day 4 completed successfully.
